# 🚗 Polynomial Regression — UCI Auto MPG Dataset
### IEEE SSCS | AI Sub-Team | Session 6

**Goal:** Predict a car's fuel efficiency (MPG) from features like horsepower, weight, and displacement using polynomial regression.

---


In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (9, 5),
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')
print("✅ All libraries imported successfully!")


---
## Step 1 — Data Loading & Exploration

We load the **UCI Auto MPG** dataset and inspect it for missing values, outliers, and distribution shape.


In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
col_names = ['mpg','cylinders','displacement','horsepower',
             'weight','acceleration','model_year','origin','name']

df = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data',
    names=col_names, sep=r'\s+', na_values='?'
)

print(f"Shape: {df.shape}")
df.head()


In [ ]:
# ── Basic statistics ──────────────────────────────────────────────────────────
print("=== Missing values ===")
print(df.isnull().sum())
print()
print("=== Descriptive statistics ===")
df.describe()


In [ ]:
# ── Clean: drop rows with missing horsepower (only 6 rows) ───────────────────
df.dropna(subset=['horsepower'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Shape after dropping NaN: {df.shape}")

# ── EDA plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Target distribution
axes[0].hist(df['mpg'], bins=25, color='steelblue', edgecolor='white')
axes[0].set(title='MPG Distribution (Target)', xlabel='MPG', ylabel='Count')

# Scatter: horsepower vs MPG
axes[1].scatter(df['horsepower'], df['mpg'], alpha=0.4, color='tomato', s=18)
axes[1].set(title='Horsepower vs MPG', xlabel='Horsepower', ylabel='MPG')

# Scatter: weight vs MPG
axes[2].scatter(df['weight'], df['mpg'], alpha=0.4, color='seagreen', s=18)
axes[2].set(title='Weight vs MPG', xlabel='Weight', ylabel='MPG')

plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight')
plt.show()
print("📊 Observations: MPG is right-skewed. Higher horsepower/weight → lower MPG (negative correlation).")


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
numeric_cols = ['mpg','cylinders','displacement','horsepower','weight','acceleration','model_year']
plt.figure(figsize=(8, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()


---
## Step 2 — Linear Baseline

We start with **horsepower** as the single predictor (strongest individual correlation with MPG).  
This linear model is our benchmark — every polynomial model must beat it.


In [ ]:
# ── Feature / target setup ────────────────────────────────────────────────────
X = df[['horsepower']].values
y = df['mpg'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

# ── Metric helper ─────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label='Model'):
    mse   = mean_squared_error(y_true, y_pred)
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    # MAPE — guard against zero targets
    mask  = y_true != 0
    mape  = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    r2    = r2_score(y_true, y_pred)
    return {'Model': label, 'MSE': round(mse,3), 'MAE': round(mae,3),
            'RMSE': round(rmse,3), 'MAPE%': round(mape,2), 'R²': round(r2,4)}

# ── Linear baseline ───────────────────────────────────────────────────────────
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lin = lin_model.predict(X_test)

baseline_metrics = compute_metrics(y_test, y_pred_lin, label='Linear Baseline')
results = [baseline_metrics]          # we'll keep adding rows here

print("\n=== Linear Baseline Metrics ===")
pd.DataFrame([baseline_metrics])


In [ ]:
# ── Visualise linear fit ──────────────────────────────────────────────────────
x_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, alpha=0.5, s=20, label='Test data')
plt.plot(x_range, lin_model.predict(x_range), color='crimson', lw=2, label='Linear fit')
plt.xlabel('Horsepower'); plt.ylabel('MPG')
plt.title('Linear Regression — Horsepower vs MPG')
plt.legend(); plt.tight_layout()
plt.savefig('linear_fit.png', bbox_inches='tight')
plt.show()


---
## Step 3 — Polynomial Feature Engineering

We expand the feature space with `PolynomialFeatures` at degrees 2, 3, 4, and 5.  
We track **training vs. validation RMSE** to spot the overfitting knee.


In [ ]:
# ── Polynomial regression loop ────────────────────────────────────────────────
degrees = [2, 3, 4, 5]
train_rmse, val_rmse, poly_models = [], [], {}

for deg in degrees:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr',     LinearRegression())
    ])
    pipe.fit(X_train, y_train)
    poly_models[deg] = pipe

    tr_rmse = np.sqrt(mean_squared_error(y_train, pipe.predict(X_train)))
    te_rmse = np.sqrt(mean_squared_error(y_test,  pipe.predict(X_test)))
    train_rmse.append(tr_rmse)
    val_rmse.append(te_rmse)

    m = compute_metrics(y_test, pipe.predict(X_test), label=f'Poly deg {deg}')
    results.append(m)
    print(f"Degree {deg}:  Train RMSE={tr_rmse:.3f}  |  Test RMSE={te_rmse:.3f}")


In [ ]:
# ── Training vs Validation RMSE curve ────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(degrees, train_rmse, 'o-', label='Train RMSE', color='steelblue')
plt.plot(degrees, val_rmse,   's-', label='Test RMSE',  color='tomato')
plt.xlabel('Polynomial Degree'); plt.ylabel('RMSE')
plt.title('Overfitting Diagnosis — RMSE vs Degree')
plt.xticks(degrees); plt.legend(); plt.tight_layout()
plt.savefig('rmse_vs_degree.png', bbox_inches='tight')
plt.show()

print("\n📌 Observation: Test RMSE decreases from degree 1→2, then slowly rises again — "
      "overfitting starts at degree 3-4.")


In [ ]:
# ── Visualise all polynomial curves ──────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.scatter(X_test, y_test, alpha=0.4, s=15, color='gray', label='Test data')

colors = ['steelblue','seagreen','orange','crimson']
for deg, col in zip(degrees, colors):
    plt.plot(x_range, poly_models[deg].predict(x_range),
             lw=2, label=f'Degree {deg}', color=col)

plt.xlabel('Horsepower'); plt.ylabel('MPG')
plt.title('Polynomial Fits — All Degrees')
plt.legend(); plt.tight_layout()
plt.savefig('poly_curves.png', bbox_inches='tight')
plt.show()


---
## Step 4 — Regularization: Ridge (L2) & Lasso (L1)

We apply regularization to **degree 3** (the sweet spot before clear overfitting).  
`GridSearchCV` with 5-fold CV finds the best `alpha` for each.


In [ ]:
# ── Prepare degree-3 polynomial features ─────────────────────────────────────
best_deg = 3
poly_tf   = PolynomialFeatures(degree=best_deg, include_bias=False)
scaler    = StandardScaler()

X_train_poly = scaler.fit_transform(poly_tf.fit_transform(X_train))
X_test_poly  = scaler.transform(poly_tf.transform(X_test))

alphas = {'alpha': np.logspace(-3, 3, 50)}

# ── Ridge ─────────────────────────────────────────────────────────────────────
ridge_cv = GridSearchCV(Ridge(), alphas, cv=5, scoring='neg_root_mean_squared_error')
ridge_cv.fit(X_train_poly, y_train)
best_ridge = ridge_cv.best_estimator_
print(f"Best Ridge alpha : {ridge_cv.best_params_['alpha']:.4f}")

# ── Lasso ─────────────────────────────────────────────────────────────────────
lasso_cv = GridSearchCV(Lasso(max_iter=10000), alphas, cv=5,
                         scoring='neg_root_mean_squared_error')
lasso_cv.fit(X_train_poly, y_train)
best_lasso = lasso_cv.best_estimator_
print(f"Best Lasso alpha : {lasso_cv.best_params_['alpha']:.4f}")

# ── Collect metrics ───────────────────────────────────────────────────────────
results.append(compute_metrics(y_test, best_ridge.predict(X_test_poly), 'Ridge (deg 3)'))
results.append(compute_metrics(y_test, best_lasso.predict(X_test_poly), 'Lasso (deg 3)'))
print("\nRegularized models evaluated ✅")


In [ ]:
# ── Ridge CV score vs alpha ───────────────────────────────────────────────────
alpha_vals   = ridge_cv.cv_results_['param_alpha'].data.astype(float)
ridge_scores = -ridge_cv.cv_results_['mean_test_score']   # back to positive RMSE
lasso_scores = -lasso_cv.cv_results_['mean_test_score']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, label, color in zip(
        axes, [ridge_scores, lasso_scores], ['Ridge', 'Lasso'], ['steelblue','tomato']):
    ax.semilogx(alpha_vals, scores, color=color, lw=2)
    ax.set(xlabel='Alpha (log scale)', ylabel='CV RMSE', title=f'{label} — Alpha Tuning')
    ax.axvline(x=(ridge_cv if label=='Ridge' else lasso_cv).best_params_['alpha'],
               color='black', linestyle='--', label=f"Best α")
    ax.legend()
plt.tight_layout()
plt.savefig('regularization_alpha.png', bbox_inches='tight')
plt.show()


---
## Step 5 — Metrics Deep Dive

All five metrics compared across every model:

| Metric | Meaning | Lower = Better? |
|--------|---------|-----------------|
| **MSE** | Mean Squared Error — penalises large errors heavily | ✅ |
| **MAE** | Mean Absolute Error — robust to outliers | ✅ |
| **RMSE** | √MSE — same unit as target, interpretable | ✅ |
| **MAPE** | % error relative to true value — scale-free | ✅ |
| **R²** | Variance explained by the model (0–1) | ❌ (higher = better) |


In [ ]:
# ── Full comparison table ─────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
print("=== Model Comparison Table ===\n")
print(results_df.to_string())
results_df


In [ ]:
# ── RMSE bar chart ────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
bars = plt.bar(results_df.index, results_df['RMSE'],
               color=sns.color_palette('tab10', len(results_df)))
plt.xticks(rotation=25, ha='right')
plt.ylabel('RMSE'); plt.title('RMSE Comparison — All Models')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{bar.get_height():.2f}", ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('rmse_comparison.png', bbox_inches='tight')
plt.show()


---
## Step 6 — Residual Analysis

We plot residuals **(ŷ − y)** vs. fitted values for the best model.  
**Ideal:** residuals scatter randomly around zero with no pattern.  
**Funnel shape:** heteroscedasticity — variance grows with prediction magnitude.


In [ ]:
# ── Best model: Ridge (degree 3) ─────────────────────────────────────────────
y_pred_best  = best_ridge.predict(X_test_poly)
residuals    = y_pred_best - y_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Residuals vs fitted
axes[0].scatter(y_pred_best, residuals, alpha=0.5, s=20, color='steelblue')
axes[0].axhline(0, color='red', lw=1.5, linestyle='--')
axes[0].set(xlabel='Fitted Values (ŷ)', ylabel='Residual (ŷ − y)',
            title='Residuals vs Fitted Values')

# Histogram of residuals
axes[1].hist(residuals, bins=25, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', lw=1.5, linestyle='--')
axes[1].set(xlabel='Residual', ylabel='Count', title='Residual Distribution')

plt.tight_layout()
plt.savefig('residuals.png', bbox_inches='tight')
plt.show()

print(f"Residual mean  : {residuals.mean():.4f}  (should be ≈ 0)")
print(f"Residual std   : {residuals.std():.4f}")


---
## Step 7 — Insights & Conclusions

### 🔍 Key Observations

**1. Non-linearity is real, but modest.**  
The jump from Linear → Degree 2 gives a meaningful RMSE improvement, confirming a curved relationship between horsepower and MPG. Beyond degree 3, gains plateau and overfitting begins (training RMSE keeps dropping while test RMSE stagnates or rises).

**2. Regularization helps at higher degrees.**  
Ridge (L2) at degree 3 matches or slightly beats unregularized polynomial models while keeping coefficients stable. Lasso (L1) achieves similar results and can zero out redundant higher-order terms, acting as automatic feature selection.

**3. RMSE is the most useful metric here.**  
Since MPG is a continuous, real-world quantity in familiar units, RMSE gives interpretable error ("off by ~3 MPG on average"). MAPE is useful for communicating % error to non-technical stakeholders. R² confirms how much variance the model explains (~0.70–0.75 for our best models). MSE is good for optimisation but hard to interpret in raw form.

### 🏆 Best Model
**Ridge Regression at Degree 3** — lowest or near-lowest test RMSE with stable, regularised coefficients.

### 📌 What the complexity tells us about the data
A degree-2 or degree-3 polynomial suffices for horsepower→MPG. The relationship is moderately curved (not linear, not wildly complex). Adding more features (weight, displacement) would likely improve the model more than increasing polynomial degree.

---
*Notebook by — IEEE SSCS AI Sub-Team | Session 6*
